# 05 — Training: Trainer + TrainedBundle + W&B opt-in

Notebook 04 produced three concrete regressors and persisted each to a model directory. This notebook adds the **orchestration layer** that pairs every fitted model with the data it was trained against, scores it on a held-out fold, and packages everything into a single self-describing artifact.

## What this notebook contributes

```text
Trainer
    fit(X_train, y_train) → score(val) → bundle.save(dir)
    optional: wandb.init → log_metrics → log_artifact

TrainedBundle             (frozen dataclass)
    regressor              # the fitted model
    feature_spec           # the recipe that produced its inputs
    metadata               # provenance + scores (see below)
    source_manifest        # the source TrainingDataset's manifest, verbatim

TrainingMetadata          (embedded inside the bundle)
    created_at_utc, susse_version, git_sha
    holdout_label, splitter_name        ← how the val fold was constructed
    n_train_rows, n_val_rows
    train_metrics, val_metrics          ← ScoreSet (mae, rmse, r2, n_rows)
    baseline_metrics                    ← raw NASA / CAMS scores on the same val fold

Splitter   (ABC)
    name: str                              # configuration identifier
    split(processed) → (train_idx, val_idx)
```

The bundle is **fully self-describing**: dropping the directory on a new machine and calling `load_bundle(dir)` gives you back a fitted regressor plus everything needed to reproduce its inputs (the `FeatureSpec`) and its lineage (the source dataset's manifest, verbatim).

## Two architectural decisions baked in

* **W&B is strict opt-in.** A default `Trainer()` performs no W&B calls. Set `TrainerConfig(wandb_project="susse")` to activate `wandb.init → log_metrics → log_artifact`. Notebook re-runs don't accidentally create runs.
* **The bundle embeds the source dataset's manifest verbatim.** That's what makes a bundle self-describing — you don't need the original W&B dataset artifact (or the warehouse) to understand what data went into the model. The manifest does carry a content-hash, so if the dataset was edited after training, the lineage is still detectable.

## Splitters

The Trainer accepts any `Splitter` ABC instance. Five concrete splitters ship in `susse.training`:

* `RandomSplitter(val_fraction, random_state)` — uniform shuffled holdout.
* `TemporalSplitter(split_date)` — every row on or after a date goes to val.
* `StationLOSOSplitter(held_out_station)` — leave-one-station-out.
* `SpatialSpreadHoldoutSplitter(n_holdout)` — hold out the `n` stations with maximum pairwise haversine spread.
* `SpatialBlockSplitter(val_blocks, block_column)` — hold out named categorical blocks (countries, climate zones).

This notebook uses `StationLOSOSplitter`; NB 06 walks through all five with comparable folds.

In [ ]:
# ============================================================
# Colab bootstrap (no-op when run locally).
# ------------------------------------------------------------
# First-time setup on Colab:
#   1. Create a GitHub Personal Access Token (PAT) at
#      https://github.com/settings/tokens with `repo` scope.
#      The repository is private, so the clone needs this token
#      (or an SSH key Colab knows about, which is more fiddly).
#   2. Add the token under Tools → Secrets in Colab with name
#      `GITHUB_PAT` and toggle "Notebook access" on.
#   3. Sign in with a Google account that has BigQuery read access
#      to `solar-irradiation-estimation` when prompted.
# ============================================================
import os
import sys

if "google.colab" in sys.modules:
    REPO = "Marconi-Lab/Solar_irradiation"
    BRANCH = "jm/add_model"

    if not os.path.exists("/content/Solar_irradiation/.git"):
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_PAT")
            clone_url = f"https://{token}@github.com/{REPO}.git"
            print("Cloning with Colab secret 'GITHUB_PAT'.")
        except Exception:
            clone_url = f"git@github.com:{REPO}.git"
            print(
                "Colab secret 'GITHUB_PAT' not set — trying SSH. If the "
                "clone fails, follow the PAT setup steps above and re-run."
            )
        !git clone -q -b {BRANCH} {clone_url} /content/Solar_irradiation

    %cd /content/Solar_irradiation
    !pip install -q -e . 2>&1 | tail -3

    from google.colab import auth
    auth.authenticate_user()
    !gcloud config set project solar-irradiation-estimation 2>/dev/null
    print("Colab setup complete.")

### Inputs, outputs, and prerequisites

| | |
|---|---|
| **Inputs** | The canonical `TrainingDataset` snapshot from NB 02 at `data/training_snapshots/susse_training_demo_v1-2024/`. Same v1 `FeatureSpec` from NB 03 / NB 04 is reapplied inline so this notebook is self-contained. |
| **Outputs** | A bundle directory per trained model under `data/bundles/_nb05_{mean_baseline,random_forest,linear}/`, each with `model/`, `feature_spec.json`, `metadata.json`, `source_manifest.json`. |
| **Prereqs** | None for offline use — the snapshot loader works without GCP or W&B. |
| **W&B** | Strict opt-in: `WANDB_PROJECT = None` by default near the bottom of the notebook. Set it to a string and `wandb login` to enable artifact logging + lineage. |
| **Consumed by** | NB 06 (evaluation) replaces the inline LOSO splitter with the full splitter abstraction. NB 07 (inference) loads bundles via `load_bundle(...)` to serve predictions. |

## 0 — Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

src_path = (Path.cwd() / "../../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Python", sys.version.split()[0])

## 1 — Load the preprocessed dataset

Same loader path as NB 03 / NB 04. The v1 `FeatureSpec` re-applies the same recipe so this notebook is self-contained — you don't need to have NB 03's output kernel-resident.

In [ ]:
from susse.datasets import load_snapshot
from susse.preprocessing import (
    ClearSkyIndexFeature,
    CyclicalDayOfYearFeature,
    FeatureSpec,
    Preprocessor,
)

SNAPSHOT_DIR = (Path.cwd() / "../../data/training_snapshots/susse_training_demo_v1-2024").resolve()
dataset = load_snapshot(SNAPSHOT_DIR)
print(f"Loaded {dataset.manifest.name} {dataset.manifest.version}: "
      f"{dataset.manifest.n_rows} rows x {dataset.manifest.n_cols} cols")

spec = FeatureSpec(
    target_column="y_ghi_kwh_m2_day",
    feature_columns=(
        "nasa_aod_550",
        "nasa_precipitable_water",
        "nasa_cloud_amount",
        "nasa_clearness_index",
        "sat_ghi_nasa_kwh_m2_day",
        "sat_ghi_cams_kwh_m2_day",
    ),
    derived_features=(
        ClearSkyIndexFeature(
            ghi_column="sat_ghi_cams_kwh_m2_day",
            ghi_clear_column="cams_ghi_clear",
            output_column="kt_cams",
        ),
        CyclicalDayOfYearFeature(),
    ),
)

processed = Preprocessor(spec).apply(dataset)
print(f"Processed: {processed.n_rows} rows x {processed.n_features} features")

## 2 — Define a splitter

Same station-LOSO rule NB 04 had inlined, now expressed as a `Splitter` ABC instance. The Trainer accepts any `Splitter` subclass, so swapping in any of NB 06's other strategies (random / temporal / spatial-spread / spatial-block) is a one-line change.

`StationLOSOSplitter.auto_pick_largest(processed)` is a classmethod factory that picks the held-out station deterministically (most rows; alphabetic tiebreak) so a re-run produces an identical val fold.

In [ ]:
from susse.training import StationLOSOSplitter

splitter = StationLOSOSplitter.auto_pick_largest(processed)
held_out_station = splitter.held_out_station

print(f"Held-out station: {held_out_station!r}")
print(f"Splitter name:    {splitter.name!r}")

# Sanity-check the split shape.
train_idx, val_idx = splitter(processed)
print(f"Train: {len(train_idx):,} rows from "
      f"{processed.df.loc[train_idx, 'location'].nunique()} stations")
print(f"Val:   {len(val_idx):,} rows from {held_out_station} only")

## 3 — Train one model end-to-end

`Trainer().train(...)` takes the preprocessed dataset, the typed params, the splitter, and a free-form `holdout_label` string that ends up in the bundle's `metadata.json`. Passing `bundle_dest=` writes the bundle to disk after training; omit it for an in-memory bundle.

W&B isn't touched here — `Trainer()` constructed without a config performs zero W&B calls.

In [ ]:
from susse.models import RandomForestParams
from susse.training import Trainer

BUNDLES_ROOT = (Path.cwd() / "../../data/bundles").resolve()
BUNDLES_ROOT.mkdir(parents=True, exist_ok=True)

rf_dest = BUNDLES_ROOT / "_nb05_random_forest"
trainer = Trainer()
rf_bundle = trainer.train(
    processed=processed,
    params=RandomForestParams(n_estimators=200, max_depth=12, random_state=42),
    splitter=splitter,
    holdout_label=f"station-LOSO:{held_out_station}",
    bundle_dest=rf_dest,
)

print(f"Trained: {type(rf_bundle.regressor).__name__}")
print(f"Bundle:  {rf_dest}")
print(f"Val MAE: {rf_bundle.metadata.val_metrics.mae:.4f}  "
      f"RMSE: {rf_bundle.metadata.val_metrics.rmse:.4f}  "
      f"R2:   {rf_bundle.metadata.val_metrics.r2:.4f}")

### Bundle on-disk layout

Every bundle holds four pieces. The model subdir is whatever `BaseRegressor.save()` writes (the same layout NB 04 demonstrated); the other three are JSON, by design — `cat metadata.json` is a useful debug operation.

In [ ]:
def show_tree(root: Path, depth: int = 2) -> None:
    for path in sorted(root.rglob("*")):
        if len(path.relative_to(root).parts) > depth:
            continue
        rel = path.relative_to(root)
        size = path.stat().st_size if path.is_file() else 0
        kind = "/" if path.is_dir() else f"  ({size:>11,} bytes)"
        print(f"  {rel}{kind}")

show_tree(rf_dest)

## 4 — Roundtrip: `load_bundle` → predict

The whole point of bundles: hand `load_bundle(dir)` to inference code with no compile-time knowledge of which model flavour was trained, and get back a fitted regressor whose predictions match the original byte-for-byte (modulo joblib's ~1-ulp drift on linear coefficients).

In [ ]:
from susse.training import load_bundle

restored = load_bundle(rf_dest)

# Same predictions as the in-memory bundle, on the val fold.
X_val = processed.X().loc[val_idx]
np.testing.assert_allclose(
    restored.regressor.predict(X_val).values,
    rf_bundle.regressor.predict(X_val).values,
    rtol=1e-10, atol=1e-10,
)
# All four pieces survived the roundtrip.
assert restored.feature_spec == rf_bundle.feature_spec
assert restored.metadata == rf_bundle.metadata
assert restored.source_manifest == rf_bundle.source_manifest
print("Roundtrip OK.")
print(f"  splitter_name:    {restored.metadata.splitter_name}")
print(f"  holdout_label:    {restored.metadata.holdout_label}")
print(f"  source dataset:   "
      f"{restored.source_manifest.name} {restored.source_manifest.version}")
print(f"  source content:   {restored.source_manifest.content_hash[:16]}...")

## 5 — Train all three models, compare against the satellite baselines

The Trainer also scores **the raw NASA / CAMS satellite estimates against the same val fold**, recorded in `metadata.baseline_metrics`. This gives every bundle a built-in answer to *"is the model worth deploying over just using the satellite?"* without re-running NB 04's inline scoring.

The trained-model rows below should beat both satellites on MAE / RMSE; the satellite rows are recorded inside *every* bundle, so this comparison is reproducible from any bundle directory alone.

In [ ]:
from susse.models import LinearParams, MeanBaselineParams

bundles: dict[str, "TrainedBundle"] = {}
for label, params in [
    ("mean_baseline", MeanBaselineParams()),
    ("random_forest", RandomForestParams(n_estimators=200, max_depth=12, random_state=42)),
    ("linear", LinearParams(with_scaling=True)),
]:
    dest = BUNDLES_ROOT / f"_nb05_{label}"
    bundles[label] = trainer.train(
        processed=processed,
        params=params,
        splitter=splitter,
        holdout_label=f"station-LOSO:{held_out_station}",
        bundle_dest=dest,
    )

# Tabulate val metrics for the trained models PLUS each bundle's
# embedded baseline metrics. We pull the baselines from the first
# bundle (they're identical across all three since the val fold is
# the same).
rows = []
for label, bundle in bundles.items():
    s = bundle.metadata.val_metrics
    rows.append({"model": label, "mae": s.mae, "rmse": s.rmse, "r2": s.r2,
                 "n_rows": s.n_rows})
for col, s in bundles["random_forest"].metadata.baseline_metrics.items():
    rows.append({"model": col, "mae": s.mae, "rmse": s.rmse, "r2": s.r2,
                 "n_rows": s.n_rows})

scores = pd.DataFrame(rows).set_index("model")
scores

## 6 — Optional: log to W&B

Setting `WANDB_PROJECT` to a string activates the full opt-in path. The Trainer:

1. Calls `wandb.init(job_type="training", config={params + manifest snippet + git_sha + splitter_name + holdout_label})`.
2. Logs the eight scalar metrics (`train/mae`, `val/r2`, `baseline/<col>/mae`, …) in one `run.log({...})`.
3. Calls `run.use_artifact(dataset_artifact_ref)` if you pass one — that's how W&B's lineage graph learns *this model used that dataset*.
4. Calls `run.log_artifact(bundle_dir, type="trained_model")` so the on-disk bundle becomes a versioned W&B artifact.

`WANDB_PROJECT = None` by default so re-running the notebook never spams runs.

In [ ]:
from susse.training import TrainerConfig

WANDB_PROJECT = None  # Set to e.g. "susse" to enable.
WANDB_ENTITY = None   # None uses your default `wandb login` entity.
DATASET_ARTIFACT_REF = None  # e.g. "<entity>/susse/susse_training_demo:v1-2024"

if WANDB_PROJECT is not None:
    wb_trainer = Trainer(TrainerConfig(
        wandb_project=WANDB_PROJECT,
        wandb_entity=WANDB_ENTITY,
        wandb_tags=("nb05", "demo"),
    ))
    rf_bundle_wb = wb_trainer.train(
        processed=processed,
        params=RandomForestParams(n_estimators=200, max_depth=12, random_state=42),
        splitter=splitter,
        holdout_label=f"station-LOSO:{held_out_station}",
        bundle_dest=BUNDLES_ROOT / "_nb05_random_forest_wb",
        dataset_artifact_ref=DATASET_ARTIFACT_REF,
    )
    print("W&B run completed and artifact logged.")
else:
    print("WANDB_PROJECT=None - skipping W&B. Set it to a string to enable.")

## 7 — Inference reconstruction demo

The portal-side use case: someone hands you a bundle directory and a fresh raw row from the warehouse. Without knowing which model flavour was trained, you should be able to reproduce inference identically.

Below: drop a single row from the held-out station's `dataset.df` (raw warehouse columns), apply the bundle's embedded `FeatureSpec` via `Preprocessor`, and ask the bundle's regressor for a prediction. The fact that the bundle carries its `feature_spec` is what makes this work — without it, you'd have no way to know which derived features (like `kt_cams`) the model expects.

In [ ]:
# Pick one raw row from the warehouse-shaped TrainingDataset.
raw_row = dataset.df.loc[dataset.df["location"] == held_out_station].head(1)
print("Raw row:")
print(raw_row.T)

# Wrap it in a single-row TrainingDataset-like input. We re-use the
# loaded TrainingDataset's manifest because the apply step only needs
# columns that are present in raw_row.
from susse.datasets import TrainingDataset
single_row_dataset = TrainingDataset(df=raw_row.reset_index(drop=True),
                                      manifest=dataset.manifest)

bundle = load_bundle(rf_dest)
processed_one = Preprocessor(bundle.feature_spec).apply(single_row_dataset)
pred = bundle.regressor.predict(processed_one.X())
truth = processed_one.y().iloc[0]
print(f"\nGround truth: {truth:.3f} kWh/m2/day")
print(f"Prediction:   {pred.iloc[0]:.3f} kWh/m2/day")
print(f"Error:        {pred.iloc[0] - truth:+.3f}")

## What's next

* **NB 06 — `06_evaluation.ipynb`** — walks through the full `Splitter` family (random / temporal / station-LOSO / spatial-spread / spatial-block) and the `Evaluator` class that compares predictions against multiple baselines under multiple metrics.
* **NB 07 — `07_inference.ipynb`** — the portal-facing API: `load_bundle(...)` plus a `predict(lat, lon, date_range)` helper that does the warehouse query → preprocess → predict pipeline.

The discipline going forward:

| Change | What ripples |
|---|---|
| New splitter strategy | New `Splitter` subclass; Trainer signature unchanged. |
| New model kind (e.g. XGBoost) | New `ModelKind` member + params + regressor (NB 04 covered the recipe). Trainer + bundle are kind-agnostic. |
| Dataset re-materialised (new ground truth, etc.) | Re-train against the new snapshot; the bundle's embedded `source_manifest.content_hash` distinguishes it from prior bundles. |
| W&B logging convention change | Update `Trainer._log_to_wandb`; the offline path is unaffected. |

The bundle is the unit of inference. If something here feels missing — a metric you'd want recorded, a provenance field that should travel with the artifact — extend `TrainingMetadata`, not the call site.